In [ ]:
import pandas as pd

In [ ]:
orders_1=pd.read_csv('../output/data_clean.csv')
orders_1['date']=pd.to_datetime(orders_1['date']).dt.normalize()


In [ ]:
#获得用户的新增日期

In [ ]:
user_first=orders_1.groupby('user_id')["date"].min().reset_index()#groupby执行后会以user_id设置为索引，reset_index将索引变为普通列

In [ ]:
user_first.columns=["user_id","first_date"]#设置新表列名

In [ ]:
df=orders_1.merge(user_first,on="user_id",how="left")#将含有新增日期的新表与原表进行连接

In [ ]:
#merge(要合并的表，匹配列，匹配方式)

In [ ]:
df["date"]=pd.to_datetime(df["date"])#将格式确定为日期，防止为字符串而无法计算

In [ ]:
df["first_date"]=pd.to_datetime(df["first_date"])

In [ ]:
df["diff_day"]=(df["date"]-df["first_date"]).dt.days#不要少()这个括号，.dt.days取出数字天数

In [ ]:
#diff_day = 0 → 用户新增的当天产生行为
#diff_day = 1 → 用户在【新增的次日】回来产生行为

In [ ]:
daily_new=df[df["diff_day"]==0].groupby("first_date")["user_id"].nunique().reset_index()
#nunique()计算出去重后个数，不可用count()因为会重复计数

In [ ]:
daily_new.columns=["stat_date","new_user_count"]

In [ ]:
daily_back=df[df["diff_day"]==1].groupby("first_date")["user_id"].nunique().reset_index()

In [ ]:
daily_back.columns=["stat_date","back_user_count"]

In [ ]:
ret_result=daily_new.merge(daily_back,on="stat_date",how="left")##合并分子分母

In [ ]:
ret_result['is_observable']=ret_result['stat_date']+pd.Timedelta(days=1)<=orders_1['date'].max()
ret_result.loc[ret_result['is_observable'],'back_user_count']=ret_result.loc[ret_result['is_observable'],'back_user_count'].fillna(0)
ret_result.loc[~ret_result['is_observable'],'back_user_count']=float('nan')


In [ ]:
ret_result['next_day_retention']=ret_result['back_user_count']/ret_result['new_user_count']
ret_result.to_csv('../output/next_day_retention_revised.csv',index=False,encoding='utf-8-sig')


In [ ]:
ret_result